# AsyncFlow — MMc split Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **multi-server** scenario compatible with **M/M/c** assumptions in the case of n parallel M/M/1
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)

> Tip: run this notebook from your project **root folder**.


In [17]:
import sys, importlib


for m in list(sys.modules):
    if m.startswith("asyncflow"):
        del sys.modules[m]


from asyncflow import AsyncFlow, SimulationRunner
from asyncflow.analysis import MMc, ResultsAnalyzer
from asyncflow.components import (
    Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
)
from asyncflow.settings import SimulationSettings

import simpy



In [18]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
from asyncflow.settings import SimulationSettings
from asyncflow.analysis import  ResultsAnalyzer, SweepAnalyzer, MMc
from asyncflow.enums import Distribution

print("Imports OK.")

Imports OK.


## 1) Build an M/M/c split-friendly scenario

* **Multiple identical servers with exponential CPU service**
  Topology includes **\$c \geq 2\$ identical servers**, each exposing exactly **one endpoint** with exactly **one CPU-bound step**.
  Service times follow an **Exponential** distribution with mean \$E\[S]\$ (service rate \$\mu = 1/E\[S]\$). No RAM/IO steps are included in the pipeline.

* **Load balancer with round-robin dispatch**
  A **single load balancer** is required when \$c > 1\$. It splits arrivals **randomly** across servers, so each server has its own local queue.
  This corresponds to a **split M/M/c** model, not the textbook pooled queue.

* **“Poisson arrivals” via the generator**
  

  

---

```mermaid
graph LR;
    rqs1["<b>RqsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    lb1["<b>LoadBalancer</b><br/>id: lb-1<br/>Policy: round_robin"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]
    app2["<b>Server</b><br/>id: app-2<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-lb<br/>Latency: 0.0001" --> lb1;
    lb1 -- "Dispatch<br/>Edge: lb-app1<br/>Latency: 0.0001" --> app1;
    lb1 -- "Dispatch<br/>Edge: lb-app2<br/>Latency: 0.0001" --> app2;
    app1 -- "Response<br/>Edge: app1-client<br/>Latency: 0.0001" --> client1;
    app2 -- "Response<br/>Edge: app2-client<br/>Latency: 0.0001" --> client1;
```

---

⚠️ **Note on model scope**
This scenario currently represents a **split M/M/c with random dispatch**.
The **textbook M/M/c (Erlang-C)** assumes a **single pooled FCFS queue feeding c servers**, which tends to give lower waiting times (no imbalance across local queues).

In a future step, we will extend AsyncFlow with a **pooled FCFS dispatcher** at the load balancer, enabling direct comparison against the textbook Erlang-C closed forms.

---



In [19]:
def build_payload():
    generator = ArrivalsGenerator(
        id="rqs-1",
        lambda_rps=30,
        model=Distribution.POISSON
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",
                "step_operation": {
                    "cpu_time": {"mean": 0.01, "distribution": "exponential"},
                },
            },
        ],
    )

    srv1 = Server(
        id="srv-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    srv2 = Server(
        id="srv-2",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    lb = LoadBalancer(
        id="lb-1",
        algorithms="random",  
        server_covered={"srv-1", "srv-2"},
    )

    edges = [
        LinkEdge(id="gen-client",  source="rqs-1",  target="client-1",),
        LinkEdge(id="client-lb",   source="client-1", target="lb-1",  ),
        LinkEdge(id="lb-srv1",     source="lb-1",   target="srv-1",   ),
        LinkEdge(id="lb-srv2",     source="lb-1",   target="srv-2",   ),
        LinkEdge(id="srv1-client", source="srv-1",  target="rqs-1",),
        LinkEdge(id="srv2-client", source="srv-2",  target="rqs-1",),
    ]

    settings = SimulationSettings(
        total_simulation_time=3600,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_arrivals_generator(generator)
        .add_client(client)
        .add_servers(srv1, srv2)
        .add_load_balancer(lb)
        .add_edges(*edges)
        .add_simulation_settings(settings)
    ).build_payload()

    return payload


## 2) Run the simulation


In [20]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")

Done.


## 3) MMc (Random) — theory vs observed comparison

If the payload violates MMc assumptions, a readable error is shown instead.
This section matches exactly what the analyzer computes **now**: a **split random model** (not the pooled FCFS/Erlang-C model). When we add **FCFS** in the LB, we’ll also expose the textbook Erlang-C formulas side-by-side.

---

## Variables (what they represent)

* **$c$**: number of *identical* servers (parallel replicas).
* **$\lambda$**: global **arrival rate** (req/s).
* **$\mu$**: **per-server** service rate (req/s) $= 1/\mathbb{E}[S]$.
* **$\rho$**: **global utilization**, $\rho = \lambda/(c\,\mu)$ (unitless).
* **$W$**: **mean time in system** (queue + service), seconds.
* **$W_q$**: **mean waiting time in queue**, seconds.
* **$L$**: **mean number in system** (queue + service), unitless.
* **$L_q$**: **mean number in queue**, unitless.
* **$\mathbb{E}[S]$**: mean **CPU service time**, seconds.

Derived (random split model):

* **$\lambda_i$**: **per-server arrival rate**, $\lambda_i = \lambda/c$.

> In the comparison table you’ll see two columns: **Theory** (closed-form) and **Observed** (estimates from the run).

---

## How we compute the **Theory** column (MMc with Round-Robin split)

1. **Predicted arrival rate**

$$
\lambda_{\text{Theory}} 
$$

2. **Predicted service rate** (from the **CPU exponential step**)

$$
\mu_{\text{Theory}} \;=\; \frac{1}{\mathbb{E}[S]}
$$

3. **Parallelism & utilization**

$$
c \;=\; \text{number of servers}, 
\qquad
\rho_{\text{Theory}} \;=\; \frac{\lambda_{\text{Theory}}}{c\,\mu_{\text{Theory}}}
$$

4. **RR split closed forms** (used by the analyzer today)

If $\rho_{\text{Theory}} \ge 1$: the system is **unstable** and
$W, W_q, L, L_q$ **diverge** (displayed as $+\infty$).

Otherwise, let $\lambda_i = \lambda_{\text{Theory}}/c$. We use:

$$
\begin{aligned}
W_{q,\text{Theory}} &= \frac{\rho_{\text{Theory}}}
                            {\mu_{\text{Theory}} - \lambda_i} \\
W_{\text{Theory}}   &= \frac{1}{\mu_{\text{Theory}}} + W_{q,\text{Theory}} \\
L_{q,\text{Theory}} &= \lambda_{\text{Theory}} \, W_{q,\text{Theory}} \\
L_{\text{Theory}}   &= \lambda_{\text{Theory}} \, W_{\text{Theory}}
\end{aligned}
$$

> 🔎 **Note:** These formulas reflect a **random split** into *c* identical M/M/1 queues (no central pool). They are **not** the Erlang-C (pooled FCFS) formulas. Once we add **FCFS** at the LB, we’ll surface the **textbook pooled M/M/c** KPIs (including $P_0$, $P_w$, etc.) alongside this random split model.

---

### How we compute the **Observed** column (from the run, M/M/1)

All estimates are computed **independently** from their own raw measurements (time-series or per-request arrays). We avoid deriving one observed KPI from another, except for explicit fallbacks when a series is unavailable.

1. **Observed arrival rate** (mean throughput over fixed windows)

   $$
   \lambda_{\text{Observed}} \;=\; \text{mean}\big(\text{RPS time series}\big)
   $$

2. **Observed time in system** (client end-to-end latency)

   $$
   W_{\text{Observed}} \;=\; \text{mean}\big(\text{client latencies}\big)
   $$

3. **Observed service rate** (from server service times)

   $$
   \overline{S}=\text{mean}(\text{service\_time}), 
   \quad
   \mu_{\text{Observed}}=
   \begin{cases}
   1/\overline{S} & \overline{S}>0\\[2pt]
   +\infty & \overline{S}=0
   \end{cases}
   $$

4. **Observed waiting time in queue** (from server ready-queue waits)

   $$
   W_{q,\text{Observed}} \;=\; \text{mean}\big(\text{waiting\_time}\big)
   $$

5. **Observed mean number in system** (from time series; **not** via Little’s Law)

   Let $L_{\text{SYSTEM}}[t]$ be the sampled series:

   $$
   L_{\text{Observed}} \;=\; \text{mean}\big(L_{\text{SYSTEM}}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $L_{\text{Observed}} = \lambda_{\text{Observed}} \, W_{\text{Observed}}$.

6. **Observed mean number in queue** (from time series; **not** via Little’s Law)

   In M/M/1 there is a single server; let $L_{q,\text{SERVER}}[t]$ be that server’s queue-length series:

   $$
   L_{q,\text{Observed}} \;=\; \text{mean}\big(L_{q,\text{SERVER}}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $L_{q,\text{Observed}} = \lambda_{\text{Observed}} \, W_{q,\text{Observed}}$.

7. **Observed utilization** (from server utilization series)

   Let $\text{UTIL}[t]\in\{0,1\}$ denote the sampled “server busy” indicator:

   $$
   \rho_{\text{Observed}} \;=\; \text{mean}\big(\text{UTIL}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $\rho_{\text{Observed}} = \lambda_{\text{Observed}} / \mu_{\text{Observed}}$ (if $\mu_{\text{Observed}}\notin\{0,+\infty\}$, else 0).

> These choices ensure each observed KPI stands on its **own** measurement (time-series or per-request data). Little’s Law based values are used **only** as safe fallbacks when the corresponding series isn’t recorded.



In [22]:
mmc = MMc()
if mmc.is_compatible(payload):
   mmc.print_comparison(payload, results)  
else:
    print("Payload is not compatible with M/M/c:")
    for reason in mmc.explain_incompatibilities(payload):
        print(" -", reason)
   


MMc (Random split) — Theory vs Observed
-------------------------------------------------------------------
sym  metric                    theory    observed        abs   rel%
-------------------------------------------------------------------
λ    Arrival rate (1/s)     30.000000   29.988611  -0.011389  -0.04
μ    Service rate (1/s)    100.000000  100.565655   0.565655   0.57
rho  Utilization             0.150000    0.149377  -0.000623  -0.42
L    Mean items in sys       0.352941    0.353297   0.000355   0.10
Lq   Mean items in queue     0.052941    0.054542   0.001601   3.02
W    Mean time in sys (s)    0.011765    0.011739  -0.000025  -0.21
Wq   Mean waiting (s)        0.001765    0.001796   0.000031   1.76
